In [1]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall", "--no-cache-dir",
    "torch==2.7.1",
    "torchvision==0.22.1",
    "torchaudio==2.7.1",
    "transformers==4.52.4",
    "accelerate==1.7.0",
    "datasets==3.6.0",
    "evaluate==0.4.3",
    "tokenizers==0.21.1"
])

0

In [2]:
import sys
import transformers
import accelerate
import datasets
import torch

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Transformers: 4.52.4
Accelerate: 1.7.0
Datasets: 3.6.0
Torch: 2.7.1+cu126
CUDA available: True


In [ ]:
# =========================================================
# 1. IMPORTS AND DRIVE SETUP
# =========================================================
import os
import csv
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch import nn
from datasets import load_dataset, Dataset

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback,
    EarlyStoppingCallback
)

from sklearn.metrics import f1_score, classification_report
from scipy.stats import pearsonr
from google.colab import drive


drive.mount("/content/drive")

os.makedirs("/content/drive/MyDrive", exist_ok=True)

LOG_FILE = "/content/drive/MyDrive/DistilBERT_SingleStep_log.csv"

TEST_EPOCH_SENTENCE_LOG = (
    "/content/drive/MyDrive/DistilBERT_SingleStep_Test_Sentence_Log_Every_Epoch.csv"
)

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow(["model", "distilbert-base-uncased"])
    writer.writerow(["learning_rate", 2e-5])
    writer.writerow(["train_batch_size", 8])
    writer.writerow(["eval_batch_size", 8])
    writer.writerow(["epochs", 3])
    writer.writerow(["sentence_log", "test sentences logged after every epoch"])
    writer.writerow([])


# =========================================================
# 2. SEED SETUP
# =========================================================
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# 3. LOAD BRIGHTER DATASET
# =========================================================
print("\nLoading BRIGHTER dataset from Hugging Face...")

CACHE_DIR = "/content/drive/MyDrive/hf_datasets_cache"

train_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="train",
    cache_dir=CACHE_DIR
)

val_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="dev",
    cache_dir=CACHE_DIR
)

test_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="test",
    cache_dir=CACHE_DIR
)

train_df = train_data.to_pandas()
val_df = val_data.to_pandas()
test_df = test_data.to_pandas()

print("\nOriginal split sizes:")
print("Train:", len(train_df))
print("Dev  :", len(val_df))
print("Test :", len(test_df))

print("\nSample data:")
print(train_df.head())


# =========================================================
# 4. LABEL CONFIGURATION
# =========================================================
EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]
LEVELS = [1, 2, 3]

LABELS = [
    f"{emotion}_{level}"
    for emotion in EMOTIONS
    for level in LEVELS
]

NUM_LABELS = len(LABELS)

print("\nLabels:")
print(LABELS)
print("Number of labels:", NUM_LABELS)


# =========================================================
# 5. CONVERT LABELS TO SINGLE-STEP FORMAT
# =========================================================
def convert_single_step_labels(df):
    df = df.copy()

    if "disgust" in df.columns:
        df = df.drop(columns=["disgust"])

    for emotion in EMOTIONS:
        df[f"{emotion}_1"] = (df[emotion] == 1).astype(int)
        df[f"{emotion}_2"] = (df[emotion] == 2).astype(int)
        df[f"{emotion}_3"] = (df[emotion] == 3).astype(int)

    return df


train_single = convert_single_step_labels(train_df)
val_single = convert_single_step_labels(val_df)
test_single = convert_single_step_labels(test_df)

print("\nSingle-step sample:")
print(train_single[["text"] + LABELS].head())


# =========================================================
# 6. COMBINE AND REDISTRIBUTE DATASET: 70 / 20 / 10
# =========================================================
full_df = pd.concat(
    [train_single, val_single, test_single],
    ignore_index=True
)

full_df = full_df[["text"] + LABELS]

full_df = full_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

n = len(full_df)

train_end = int(0.7 * n)
val_end = int(0.9 * n)

train_single = full_df[:train_end].reset_index(drop=True)
val_single = full_df[train_end:val_end].reset_index(drop=True)
test_single = full_df[val_end:].reset_index(drop=True)

print("\nRedistributed split sizes:")
print({
    "train": len(train_single),
    "val": len(val_single),
    "test": len(test_single)
})

# Keep only test texts because we log sentence-wise results only for test set
test_texts = test_single["text"].tolist()


# =========================================================
# 7. CONVERT TO HUGGING FACE DATASET
# =========================================================
train_single = Dataset.from_pandas(train_single, preserve_index=False)
val_single = Dataset.from_pandas(val_single, preserve_index=False)
test_single = Dataset.from_pandas(test_single, preserve_index=False)


# =========================================================
# 8. TOKENIZATION
# =========================================================
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )


train_single = train_single.map(tokenize_function, batched=True)
val_single = val_single.map(tokenize_function, batched=True)
test_single = test_single.map(tokenize_function, batched=True)


# =========================================================
# 9. ADD LABEL VECTOR
# =========================================================
def add_labels(example):
    example["labels"] = [float(example[label]) for label in LABELS]
    return example


train_single = train_single.map(add_labels)
val_single = val_single.map(add_labels)
test_single = test_single.map(add_labels)

train_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)


# =========================================================
# 10. METRICS
# =========================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    f1_macro = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        labels,
        preds,
        average="micro",
        zero_division=0
    )

    pearsons = []

    for i in range(labels.shape[1]):
        if np.std(labels[:, i]) == 0 or np.std(probs[:, i]) == 0:
            pearsons.append(0.0)
        else:
            p, _ = pearsonr(labels[:, i], probs[:, i])
            pearsons.append(0.0 if np.isnan(p) else float(p))

    pearson_mean = float(np.mean(pearsons))

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": pearson_mean
    }


# =========================================================
# 11. LISTS FOR PLOTTING AND LOGGING
# =========================================================
epoch_list = []

train_loss_list = []

val_loss_list = []
val_f1_macro_list = []
val_f1_micro_list = []
val_pearson_mean_list = []

test_loss_list = []
test_f1_macro_list = []
test_f1_micro_list = []
test_pearson_mean_list = []


# =========================================================
# 12. HELPER FUNCTIONS FOR SENTENCE-WISE TEST LOGGING
# =========================================================
def clean_number(x):
    """
    Removes unnecessary trailing zeros.
    Example:
    0.0000 -> 0
    0.1500 -> 0.15
    1.0000 -> 1
    """
    x = round(float(x), 4)

    if x == 0:
        return 0

    if x == 1:
        return 1

    return x


def labels_to_text(binary_labels, label_names):
    selected_labels = [
        label_names[i]
        for i, value in enumerate(binary_labels)
        if int(value) == 1
    ]

    if len(selected_labels) == 0:
        return "No Emotion"

    return ", ".join(selected_labels)


# =========================================================
# 13. CALLBACK FOR EPOCH LOGGING + TEST SENTENCE LOGGING
# =========================================================
class SaveEpochResultsCallback(TrainerCallback):
    def __init__(
        self,
        file_path,
        test_dataset,
        test_texts,
        sentence_log_path
    ):
        self.file_path = file_path
        self.test_dataset = test_dataset
        self.test_texts = test_texts
        self.sentence_log_path = sentence_log_path

        self.trainer_ref = None
        self.current_train_loss = None
        self._inside_eval = False

        # Create / reset sentence-wise test log file
        with open(self.sentence_log_path, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)

            header = [
                "epoch",
                "sentence_id",
                "sentence",
                "true_labels",
                "predicted_labels"
            ]

            for label in LABELS:
                header.append(f"prob_{label}")

            writer.writerow(header)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs and "eval_loss" not in logs:
            self.current_train_loss = float(logs["loss"])

    def write_test_sentence_log_for_epoch(
        self,
        epoch,
        true_labels,
        pred_labels,
        probabilities
    ):
        """
        Writes sentence-wise test predictions for the current epoch.
        """

        with open(self.sentence_log_path, "a", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)

            for i in range(len(self.test_texts)):
                row = [
                    epoch,
                    i,
                    self.test_texts[i],
                    labels_to_text(true_labels[i], LABELS),
                    labels_to_text(pred_labels[i], LABELS)
                ]

                for j in range(len(LABELS)):
                    row.append(clean_number(probabilities[i][j]))

                writer.writerow(row)

        print(f"Sentence-wise test predictions saved for epoch {epoch}.")

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if self._inside_eval:
            return

        if metrics is None:
            return

        self._inside_eval = True

        epoch = int(round(float(metrics.get("epoch", state.epoch))))

        train_loss = (
            self.current_train_loss
            if self.current_train_loss is not None
            else ""
        )

        val_loss = float(metrics.get("eval_loss", 0.0))
        val_f1_macro = float(metrics.get("eval_f1_macro", 0.0))
        val_f1_micro = float(metrics.get("eval_f1_micro", 0.0))
        val_pearson_mean = float(metrics.get("eval_pearson_mean", 0.0))

        test_results = self.trainer_ref.evaluate(
            eval_dataset=self.test_dataset,
            metric_key_prefix="test"
        )

        test_loss = float(test_results.get("test_loss", 0.0))
        test_f1_macro = float(test_results.get("test_f1_macro", 0.0))
        test_f1_micro = float(test_results.get("test_f1_micro", 0.0))
        test_pearson_mean = float(test_results.get("test_pearson_mean", 0.0))

        epoch_list.append(epoch)

        train_loss_list.append(train_loss)

        val_loss_list.append(val_loss)
        val_f1_macro_list.append(val_f1_macro)
        val_f1_micro_list.append(val_f1_micro)
        val_pearson_mean_list.append(val_pearson_mean)

        test_loss_list.append(test_loss)
        test_f1_macro_list.append(test_f1_macro)
        test_f1_micro_list.append(test_f1_micro)
        test_pearson_mean_list.append(test_pearson_mean)

        pred = self.trainer_ref.predict(self.test_dataset)

        logits = pred.predictions
        true_labels = pred.label_ids

        probs = 1 / (1 + np.exp(-logits))
        pred_labels = (probs >= 0.5).astype(int)

        # Sentence-wise test log for this epoch
        self.write_test_sentence_log_for_epoch(
            epoch=epoch,
            true_labels=true_labels,
            pred_labels=pred_labels,
            probabilities=probs
        )

        report_dict = classification_report(
            true_labels,
            pred_labels,
            target_names=LABELS,
            zero_division=0,
            output_dict=True
        )

        with open(self.file_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)

            writer.writerow([])
            writer.writerow([f"EPOCH {epoch}"])

            writer.writerow([
                "epoch",
                "train_loss",
                "val_loss",
                "test_loss",
                "val_f1_macro",
                "val_f1_micro",
                "test_f1_macro",
                "test_f1_micro",
                "val_pearson_mean",
                "test_pearson_mean"
            ])

            for i in range(len(epoch_list)):
                writer.writerow([
                    epoch_list[i],
                    train_loss_list[i],
                    val_loss_list[i],
                    test_loss_list[i],
                    val_f1_macro_list[i],
                    val_f1_micro_list[i],
                    test_f1_macro_list[i],
                    test_f1_micro_list[i],
                    val_pearson_mean_list[i],
                    test_pearson_mean_list[i]
                ])

            # -------------------------
            # FINAL TEST SCORES AFTER THIS EPOCH
            # -------------------------
            writer.writerow([])
            writer.writerow([f"FINAL TEST SCORES AFTER EPOCH {epoch}"])
            writer.writerow(["metric", "value"])
            writer.writerow(["test_loss", test_loss])
            writer.writerow(["test_f1_macro", test_f1_macro])
            writer.writerow(["test_f1_micro", test_f1_micro])
            writer.writerow(["test_pearson_mean", test_pearson_mean])

            # -------------------------
            # CLASSWISE RESULTS AFTER THIS EPOCH
            # -------------------------
            writer.writerow([])
            writer.writerow([f"CLASSWISE RESULTS AFTER EPOCH {epoch}"])
            writer.writerow([
                "class",
                "precision",
                "recall",
                "f1_score",
                "support"
            ])

            for class_name in LABELS:
                row = report_dict.get(class_name, {})

                writer.writerow([
                    class_name,
                    row.get("precision", ""),
                    row.get("recall", ""),
                    row.get("f1-score", ""),
                    row.get("support", "")
                ])

        print(
            f"\nEpoch {epoch} cumulative losses, final test scores, "
            f"classwise results, and sentence-wise test results saved."
        )

        self._inside_eval = False


# =========================================================
# 14. MODEL
# =========================================================
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)


# =========================================================
# 15. TRAINING ARGUMENTS
# =========================================================
training_args = TrainingArguments(
    output_dir="/content/distilbert_output",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    # Early stopping monitors validation loss
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)


# =========================================================
# 16. TRAINER
# =========================================================
callback = SaveEpochResultsCallback(
    file_path=LOG_FILE,
    test_dataset=test_single,
    test_texts=test_texts,
    sentence_log_path=TEST_EPOCH_SENTENCE_LOG
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_single,
    eval_dataset=val_single,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[
        callback,
        EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.0
        )
    ]
)

callback.trainer_ref = trainer


# =========================================================
# 17. TRAIN
# =========================================================
start = time.time()

trainer.train()

end = time.time()

print(f"\nTotal training time: {end - start:.1f} seconds")
print("Epochwise result file saved at:", LOG_FILE)
print("Test sentence-wise epoch log saved at:", TEST_EPOCH_SENTENCE_LOG)


# =========================================================
# 18. FINAL TEST EVALUATION
# =========================================================
final_test_results = trainer.evaluate(
    eval_dataset=test_single,
    metric_key_prefix="final_test"
)

print("\nFinal test results:")
print(final_test_results)


# =========================================================
# 19. DISPLAY SAMPLE TEST SENTENCE LOG
# =========================================================
test_epoch_sentence_log_df = pd.read_csv(TEST_EPOCH_SENTENCE_LOG)

print("\nSample test sentence-wise log for every epoch:")
display(test_epoch_sentence_log_df.head(10))

In [13]:
# =========================================================
# 1. IMPORTS AND DRIVE SETUP
# =========================================================
import os
import csv
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch import nn
from datasets import load_dataset, Dataset

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback,
    EarlyStoppingCallback
)

from sklearn.metrics import f1_score, classification_report
from scipy.stats import pearsonr



In [14]:
from google.colab import drive


drive.mount("/content/drive")

LOG_FILE = "/content/drive/MyDrive/DistilBERT_SingleStep_log.csv"

TRAIN_SENTENCE_LOG = "/content/drive/MyDrive/DistilBERT_SingleStep_train_sentence_log.csv"
VAL_SENTENCE_LOG = "/content/drive/MyDrive/DistilBERT_SingleStep_val_sentence_log.csv"
TEST_SENTENCE_LOG = "/content/drive/MyDrive/DistilBERT_SingleStep_test_sentence_log.csv"

os.makedirs("/content/drive/MyDrive", exist_ok=True)

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow(["model", "distilbert-base-uncased"])
    writer.writerow(["learning_rate", 2e-5])
    writer.writerow(["train_batch_size", 8])
    writer.writerow(["eval_batch_size", 8])
    writer.writerow(["epochs", 3])
    writer.writerow(["early_stopping_metric", "eval_loss"])
    writer.writerow(["early_stopping_patience", 1])
    writer.writerow([])


# =========================================================
# 2. SEED SETUP
# =========================================================
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda


In [15]:
# =========================================================
# 3. LOAD BRIGHTER DATASET
# =========================================================
print("\nLoading BRIGHTER dataset from Hugging Face...")

train_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="train"
)

val_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="dev"
)

test_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="test"
)

train_df = train_data.to_pandas()
val_df = val_data.to_pandas()
test_df = test_data.to_pandas()

print("\nOriginal split sizes:")
print("Train:", len(train_df))
print("Dev  :", len(val_df))
print("Test :", len(test_df))

print("\nSample data:")
print(train_df.head())





Loading BRIGHTER dataset from Hugging Face...

Original split sizes:
Train: 2763
Dev  : 115
Test : 2765

Sample data:
                        id                                               text  \
0  eng_train_track_b_00001                       Colorado, middle of nowhere.   
1  eng_train_track_b_00002  This involved swimming a pretty large lake tha...   
2  eng_train_track_b_00003        It was one of my most shameful experiences.   
3  eng_train_track_b_00004  After all, I had vegetables coming out my ears...   
4  eng_train_track_b_00005                        Then the screaming started.   

   anger  disgust  fear  joy  sadness  surprise  
0      0      NaN     1    0        0         1  
1      0      NaN     2    0        0         0  
2      0      NaN     1    0        3         0  
3      0      NaN     0    0        0         0  
4      0      NaN     3    0        1         2  


In [16]:
# =========================================================
# 4. LABEL CONFIGURATION
# =========================================================
EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]
LEVELS = [1, 2, 3]

LABELS = [f"{emotion}_{level}" for emotion in EMOTIONS for level in LEVELS]
NUM_LABELS = len(LABELS)

print("\nLabels:")
print(LABELS)
print("Number of labels:", NUM_LABELS)


# =========================================================
# 5. CONVERT LABELS TO SINGLE-STEP FORMAT
# =========================================================
def convert_single_step_labels(df):
    df = df.copy()

    if "disgust" in df.columns:
        df = df.drop(columns=["disgust"])

    for emotion in EMOTIONS:
        df[f"{emotion}_1"] = (df[emotion] == 1).astype(int)
        df[f"{emotion}_2"] = (df[emotion] == 2).astype(int)
        df[f"{emotion}_3"] = (df[emotion] == 3).astype(int)

    return df


train_single = convert_single_step_labels(train_df)
val_single = convert_single_step_labels(val_df)
test_single = convert_single_step_labels(test_df)

print("\nSingle-step sample:")
print(train_single[["text"] + LABELS].head())


# =========================================================
# 6. COMBINE AND REDISTRIBUTE DATASET: 70 / 20 / 10
# =========================================================
full_df = pd.concat(
    [train_single, val_single, test_single],
    ignore_index=True
)

full_df = full_df[["text"] + LABELS]

full_df = full_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

n = len(full_df)

train_end = int(0.7 * n)
val_end = int(0.9 * n)

train_single = full_df[:train_end].reset_index(drop=True)
val_single = full_df[train_end:val_end].reset_index(drop=True)
test_single = full_df[val_end:].reset_index(drop=True)

print("\nRedistributed split sizes:")
print({
    "train": len(train_single),
    "val": len(val_single),
    "test": len(test_single)
})

# Keep original sentences before converting to Hugging Face Dataset
train_texts = train_single["text"].tolist()
val_texts = val_single["text"].tolist()
test_texts = test_single["text"].tolist()


# =========================================================
# 7. CONVERT TO HUGGING FACE DATASET
# =========================================================
train_single = Dataset.from_pandas(train_single, preserve_index=False)
val_single = Dataset.from_pandas(val_single, preserve_index=False)
test_single = Dataset.from_pandas(test_single, preserve_index=False)


# =========================================================
# 8. TOKENIZATION
# =========================================================
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

train_single = train_single.map(tokenize_function, batched=True)
val_single = val_single.map(tokenize_function, batched=True)
test_single = test_single.map(tokenize_function, batched=True)


# =========================================================
# 9. ADD LABEL VECTOR
# =========================================================
def add_labels(example):
    example["labels"] = [float(example[label]) for label in LABELS]
    return example

train_single = train_single.map(add_labels)
val_single = val_single.map(add_labels)
test_single = test_single.map(add_labels)

train_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)





Labels:
['anger_1', 'anger_2', 'anger_3', 'fear_1', 'fear_2', 'fear_3', 'joy_1', 'joy_2', 'joy_3', 'sadness_1', 'sadness_2', 'sadness_3', 'surprise_1', 'surprise_2', 'surprise_3']
Number of labels: 15

Single-step sample:
                                                text  anger_1  anger_2  \
0                       Colorado, middle of nowhere.        0        0   
1  This involved swimming a pretty large lake tha...        0        0   
2        It was one of my most shameful experiences.        0        0   
3  After all, I had vegetables coming out my ears...        0        0   
4                        Then the screaming started.        0        0   

   anger_3  fear_1  fear_2  fear_3  joy_1  joy_2  joy_3  sadness_1  sadness_2  \
0        0       1       0       0      0      0      0          0          0   
1        0       0       1       0      0      0      0          0          0   
2        0       1       0       0      0      0      0          0          0   
3       

Map:   0%|          | 0/3950 [00:00<?, ? examples/s]

Map:   0%|          | 0/1128 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

Map:   0%|          | 0/3950 [00:00<?, ? examples/s]

Map:   0%|          | 0/1128 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

In [18]:
# =========================================================
# 10. METRICS
# =========================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    f1_macro = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        labels,
        preds,
        average="micro",
        zero_division=0
    )

    pearsons = []

    for i in range(labels.shape[1]):
        if np.std(labels[:, i]) == 0 or np.std(probs[:, i]) == 0:
            pearsons.append(0.0)
        else:
            p, _ = pearsonr(labels[:, i], probs[:, i])
            pearsons.append(0.0 if np.isnan(p) else float(p))

    pearson_mean = float(np.mean(pearsons))

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": pearson_mean
    }


# =========================================================
# 11. LISTS FOR PLOTTING AND LOGGING
# =========================================================
epoch_list = []

train_loss_list = []

val_loss_list = []
val_f1_macro_list = []
val_f1_micro_list = []
val_pearson_mean_list = []

test_loss_list = []
test_f1_macro_list = []
test_f1_micro_list = []
test_pearson_mean_list = []


# =========================================================
# 12. CALLBACK FOR EPOCH LOGGING
# =========================================================
class SaveEpochResultsCallback(TrainerCallback):
    def __init__(self, file_path, test_dataset):
        self.file_path = file_path
        self.test_dataset = test_dataset

        self.trainer_ref = None
        self.current_train_loss = None
        self._inside_eval = False

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs and "eval_loss" not in logs:
            self.current_train_loss = float(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if self._inside_eval:
            return

        if metrics is None:
            return

        self._inside_eval = True

        epoch = int(round(float(metrics.get("epoch", state.epoch))))

        train_loss = (
            self.current_train_loss
            if self.current_train_loss is not None
            else ""
        )

        val_loss = float(metrics.get("eval_loss", 0.0))
        val_f1_macro = float(metrics.get("eval_f1_macro", 0.0))
        val_f1_micro = float(metrics.get("eval_f1_micro", 0.0))
        val_pearson_mean = float(metrics.get("eval_pearson_mean", 0.0))

        test_results = self.trainer_ref.evaluate(
            eval_dataset=self.test_dataset,
            metric_key_prefix="test"
        )

        test_loss = float(test_results.get("test_loss", 0.0))
        test_f1_macro = float(test_results.get("test_f1_macro", 0.0))
        test_f1_micro = float(test_results.get("test_f1_micro", 0.0))
        test_pearson_mean = float(test_results.get("test_pearson_mean", 0.0))

        epoch_list.append(epoch)

        train_loss_list.append(train_loss)

        val_loss_list.append(val_loss)
        val_f1_macro_list.append(val_f1_macro)
        val_f1_micro_list.append(val_f1_micro)
        val_pearson_mean_list.append(val_pearson_mean)

        test_loss_list.append(test_loss)
        test_f1_macro_list.append(test_f1_macro)
        test_f1_micro_list.append(test_f1_micro)
        test_pearson_mean_list.append(test_pearson_mean)

        pred = self.trainer_ref.predict(self.test_dataset)

        logits = pred.predictions
        true_labels = pred.label_ids

        probs = 1 / (1 + np.exp(-logits))
        pred_labels = (probs >= 0.5).astype(int)

        report_dict = classification_report(
            true_labels,
            pred_labels,
            target_names=LABELS,
            zero_division=0,
            output_dict=True
        )

        with open(self.file_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)

            writer.writerow([])
            writer.writerow([f"EPOCH {epoch}"])

            writer.writerow([
                "epoch",
                "train_loss",
                "val_loss",
                "test_loss",
                "val_f1_macro",
                "val_f1_micro",
                "test_f1_macro",
                "test_f1_micro",
                "val_pearson_mean",
                "test_pearson_mean"
            ])

            for i in range(len(epoch_list)):
                writer.writerow([
                    epoch_list[i],
                    train_loss_list[i],
                    val_loss_list[i],
                    test_loss_list[i],
                    val_f1_macro_list[i],
                    val_f1_micro_list[i],
                    test_f1_macro_list[i],
                    test_f1_micro_list[i],
                    val_pearson_mean_list[i],
                    test_pearson_mean_list[i]
                ])

            # -------------------------
            # FINAL TEST SCORES AFTER THIS EPOCH
            # -------------------------
            writer.writerow([])
            writer.writerow([f"FINAL TEST SCORES AFTER EPOCH {epoch}"])
            writer.writerow(["metric", "value"])
            writer.writerow(["test_loss", test_loss])
            writer.writerow(["test_f1_macro", test_f1_macro])
            writer.writerow(["test_f1_micro", test_f1_micro])
            writer.writerow(["test_pearson_mean", test_pearson_mean])

            # -------------------------
            # CLASSWISE RESULTS AFTER THIS EPOCH
            # -------------------------
            writer.writerow([])
            writer.writerow([f"CLASSWISE RESULTS AFTER EPOCH {epoch}"])
            writer.writerow([
                "class",
                "precision",
                "recall",
                "f1_score",
                "support"
            ])

            for class_name in LABELS:
                row = report_dict.get(class_name, {})

                writer.writerow([
                    class_name,
                    row.get("precision", ""),
                    row.get("recall", ""),
                    row.get("f1-score", ""),
                    row.get("support", "")
                ])

        print(
            f"\nEpoch {epoch} cumulative losses, final test scores, "
            f"and classwise results saved."
        )

        self._inside_eval = False


# =========================================================
# 13. MODEL
# =========================================================
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)


# =========================================================
# 14. TRAINING ARGUMENTS WITH EARLY STOPPING ON EVAL LOSS
# =========================================================
training_args = TrainingArguments(
    output_dir="/content/distilbert_output",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    # Early stopping monitors validation loss
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)


# =========================================================
# 15. TRAINER
# =========================================================
callback = SaveEpochResultsCallback(
    file_path=LOG_FILE,
    test_dataset=test_single
)



trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_single,
    eval_dataset=val_single,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[callback,
               EarlyStoppingCallback(
            early_stopping_patience = 3,
            early_stopping_threshold=0.0
        )]
)

callback.trainer_ref = trainer


# =========================================================
# 16. TRAIN
# =========================================================
start = time.time()

trainer.train()

end = time.time()

print(f"\nTotal training time: {end - start:.1f} seconds")
print("Epochwise result file saved at:", LOG_FILE)


# =========================================================
# 17. SENTENCE-WISE CLASSIFICATION LOGGING ONLY
# =========================================================
def clean_number(x):
    """
    Removes unnecessary trailing zeros.
    Example:
    0.0000 -> 0
    0.1500 -> 0.15
    1.0000 -> 1
    """
    x = round(float(x), 4)

    if x == 0:
        return 0

    if x == 1:
        return 1

    return x


def labels_to_text(binary_labels, label_names):
    selected_labels = [
        label_names[i]
        for i, value in enumerate(binary_labels)
        if int(value) == 1
    ]

    if len(selected_labels) == 0:
        return "No Emotion"

    return ", ".join(selected_labels)


def create_sentence_classification_log(
    trainer,
    dataset,
    raw_texts,
    label_names,
    output_path,
    threshold=0.5
):
    """
    Creates sentence-wise classification log:
    sentence, true labels, predicted labels, and class probabilities.
    """

    predictions = trainer.predict(dataset)

    logits = predictions.predictions
    y_true = predictions.label_ids

    probabilities = 1 / (1 + np.exp(-logits))
    y_pred = (probabilities >= threshold).astype(int)

    rows = []

    for i in range(len(raw_texts)):
        row = {
            "sentence_id": i,
            "sentence": raw_texts[i],
            "true_labels": labels_to_text(y_true[i], label_names),
            "predicted_labels": labels_to_text(y_pred[i], label_names)
        }

        for j, label in enumerate(label_names):
            row[f"prob_{label}"] = clean_number(probabilities[i][j])

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"Saved sentence-wise classification log: {output_path}")

    return df


# =========================================================
# 18. SAVE SENTENCE-WISE LOGS FOR TRAIN, VAL, AND TEST
# =========================================================
train_sentence_log_df = create_sentence_classification_log(
    trainer=trainer,
    dataset=train_single,
    raw_texts=train_texts,
    label_names=LABELS,
    output_path=TRAIN_SENTENCE_LOG,
    threshold=0.5
)

val_sentence_log_df = create_sentence_classification_log(
    trainer=trainer,
    dataset=val_single,
    raw_texts=val_texts,
    label_names=LABELS,
    output_path=VAL_SENTENCE_LOG,
    threshold=0.5
)

test_sentence_log_df = create_sentence_classification_log(
    trainer=trainer,
    dataset=test_single,
    raw_texts=test_texts,
    label_names=LABELS,
    output_path=TEST_SENTENCE_LOG,
    threshold=0.5
)

print("\nSentence-wise train log saved at:", TRAIN_SENTENCE_LOG)
print("Sentence-wise validation log saved at:", VAL_SENTENCE_LOG)
print("Sentence-wise test log saved at:", TEST_SENTENCE_LOG)

print("\nSample train sentence-wise classification log:")
display(train_sentence_log_df.head(10))

print("\nSample validation sentence-wise classification log:")
display(val_sentence_log_df.head(10))

print("\nSample test sentence-wise classification log:")
display(test_sentence_log_df.head(10))


# =========================================================
# 19. FINAL TEST EVALUATION
# =========================================================
final_test_results = trainer.evaluate(
    eval_dataset=test_single,
    metric_key_prefix="final_test"
)

print("\nFinal test results:")
print(final_test_results)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Pearson Mean
1,0.320400,0.277255,0.000000,0.000000,0.266044
2,0.256800,0.251953,0.015089,0.024323,0.370033
3,0.231900,0.245840,0.058906,0.104462,0.384589


early stopping required metric_for_best_model, but did not find eval_loss so early stopping is disabled



Epoch 1 cumulative losses, final test scores, and classwise results saved.


early stopping required metric_for_best_model, but did not find eval_loss so early stopping is disabled


...

,sentence_id,sentence,true_labels,predicted_labels,prob_anger_1,prob_anger_2,prob_anger_3,prob_fear_1,prob_fear_2,prob_fear_3,prob_joy_1,prob_joy_2,prob_joy_3,prob_sadness_1,prob_sadness_2,prob_sadness_3,prob_surprise_1,prob_surprise_2,prob_surprise_3
0,0,"my throat is tight, my nose is numb.","fear_2, sadness_1",No Emotion,0.0294,0.0148,0.0081,0.4577,0.3219,0.0556,0.0488,0.0129,0.0063,0.3617,0.0804,0.0235,0.0443,0.0204,0.0081
1,1,"I went up on stage and sang my heart out, and ...",joy_2,joy_1,0.0395,0.0322,0.0160,0.0953,0.0254,0.0248,0.5551,0.3800,0.1480,0.0534,0.0325,0.0223,0.1454,0.0508,0.0256
2,2,Her face when her mum told her she miscarried ...,"fear_1, sadness_3",No Emotion,0.1493,0.0706,0.0407,0.3528,0.2815,0.1207,0.0287,0.0302,0.0220,0.3853,0.3910,0.3034,0.1715,0.0616,0.0266
3,3,... but my head still throbs because eventuall...,"fear_1, sadness_1",fear_2,0.0644,0.0322,0.0198,0.3723,0.5370,0.1463,0.0400,0.0144,0.0103,0.4577,0.1340,0.0533,0.0526,0.0291,0.0150
4,4,My ears bent back.,surprise_1,No Emotion,0.0209,0.0100,0.0046,0.3690,0.0759,0.0162,0.1371,0.0203,0.0066,0.1368,0.0245,0.0081,0.0551,0.0170,0.0054
5,5,I go to her pictures and see the image of what...,"joy_1, surprise_1",No Emotion,0.0297,0.0127,0.0067,0.4158,0.1383,0.0343,0.0600,0.0153,0.0063,0.1522,0.0311,0.0094,0.2680,0.0712,0.0091
6,6,"Dropping the knife, and emptying my mouth, I p...",joy_1,No Emotion,0.1895,0.0911,0.0438,0.3483,0.3868,0.1019,0.0355,0.0167,0.0112,0.3081,0.1327,0.0595,0.1522,0.0590,0.0201
7,7,"""The name of my rig is Phantom 309.",surprise_1,No Emotion,0.0489,0.0198,0.0112,0.4233,0.1334,0.0399,0.0882,0.0227,0.0109,0.1250,0.0266,0.0090,0.4934,0.1467,0.0176
8,8,its 11:30 pm in krakow and i can hardly open m...,"fear_1, sadness_1",No Emotion,0.0373,0.0160,0.0104,0.4673,0.3653,0.0748,0.0326,0.0125,0.0065,0.3722,0.1196,0.0377,0.1215,0.0404,0.0111
9,9,The entire summer is going to pass by with a b...,"fear_1, joy_1",No Emotion,0.0232,0.0129,0.0052,0.2314,0.0422,0.0158,0.3459,0.0895,0.0218,0.0698,0.0166,0.0065,0.1063,0.0307,0.0090



Sample test sentence-wise classification log:


,sentence_id,sentence,true_labels,predicted_labels,prob_anger_1,prob_anger_2,prob_anger_3,prob_fear_1,prob_fear_2,prob_fear_3,prob_joy_1,prob_joy_2,prob_joy_3,prob_sadness_1,prob_sadness_2,prob_sadness_3,prob_surprise_1,prob_surprise_2,prob_surprise_3
0,0,He changed our last name ever-so-slightly and ...,surprise_2,No Emotion,0.1604,0.0711,0.0332,0.4641,0.2087,0.0408,0.0545,0.0271,0.0147,0.2730,0.1474,0.0561,0.3727,0.1019,0.0219
1,1,They just never went away.,"fear_1, sadness_2",No Emotion,0.1188,0.0573,0.0292,0.3508,0.1772,0.0552,0.0436,0.0521,0.0262,0.3152,0.3314,0.2203,0.1318,0.0457,0.0191
2,2,"""Well I have loads, you can share mine"".",joy_1,joy_1,0.0340,0.0248,0.0108,0.0952,0.0222,0.0176,0.5293,0.2328,0.0753,0.0483,0.0268,0.0169,0.1136,0.0334,0.0157
3,3,Pretty much everyone objected to my wedding.,"anger_1, fear_1, sadness_2, surprise_1",No Emotion,0.1467,0.0761,0.0267,0.4034,0.0930,0.0198,0.0970,0.0409,0.0165,0.2043,0.0854,0.0294,0.1399,0.0389,0.0124
4,4,"I was in better bike shape last year, and had ...",No Emotion,No Emotion,0.0217,0.0163,0.0067,0.1795,0.0323,0.0140,0.4732,0.1164,0.0285,0.0796,0.0230,0.0115,0.0518,0.0192,0.0110
5,5,"Somehow, while scrubbing the left side of my f...","fear_1, surprise_1",No Emotion,0.0808,0.0314,0.0184,0.4012,0.4222,0.0955,0.0273,0.0121,0.0076,0.4374,0.1571,0.0679,0.0760,0.0302,0.0121
6,6,At about 4 i was running with a pole with one ...,"fear_2, surprise_1",No Emotion,0.0879,0.0388,0.0258,0.3699,0.4190,0.2170,0.0496,0.0208,0.0148,0.2290,0.0512,0.0223,0.3827,0.2129,0.0317
7,7,No one else in the class did.,"sadness_1, surprise_1",No Emotion,0.1138,0.0465,0.0211,0.3284,0.2362,0.0698,0.0314,0.0160,0.0105,0.1886,0.0909,0.0347,0.4599,0.1330,0.0192
8,8,My tongue had sucked it up and the hole was cl...,surprise_1,No Emotion,0.0570,0.0247,0.0109,0.4863,0.3060,0.0451,0.0393,0.0117,0.0054,0.3510,0.0746,0.0222,0.0616,0.0222,0.0077
9,9,"Because this is a karma thread, you know how t...",surprise_1,No Emotion,0.0979,0.0390,0.0173,0.4205,0.1532,0.0295,0.0562,0.0232,0.0107,0.1956,0.0824,0.0277,0.3019,0.0663,0.0120


early stopping required metric_for_best_model, but did not find eval_loss so early stopping is disabled
early stopping required metric_for_best_model, but did not find eval_loss so early stopping is disabled



Epoch 3 cumulative losses, final test scores, and classwise results saved.

Final test results:
{'final_test_loss': 0.24224986135959625, 'final_test_f1_macro': 0.07228882949923411, 'final_test_f1_micro': 0.12195121951219512, 'final_test_pearson_mean': 0.3725076804558436, 'final_test_runtime': 0.6877, 'final_test_samples_per_second': 821.572, 'final_test_steps_per_second': 103.242, 'epoch': 3.0}
